In [11]:
# use gemini build dataset
import os
import numpy as np
import numpy_financial as npf
import pandas as pd
import math
import csv
from datetime import datetime

from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

def gemini_response(model, instruction, count):
    scores = []

    for i in range(count):

        try:
            result = model.invoke(instruction)
            scores.append(int(result.content))
        except:
            pass

    if len(scores) == 0:
        score = 0
    else:
        score = sum(scores) / len(scores)

    return [date_obj.strftime('%Y%m%d'), score]  # [時間, 分數]



if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

regenerate_count = 10
# prompt_mutate_count = 100

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)
irr_ = 0

stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]


p_mutate = '深入的公司創新分析及其對預期市場影響的評估，表明股價持續上漲，反映了投資者對公司令人振奮的長期增長前景的樂觀情緒。'

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=1,
    safety_settings={
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    })


id = []
text = []
response = []

for idx, stock_id in enumerate(stock_ids):
    print(f'執行{stock_id}, {idx}')
    folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
    files = os.listdir(folder_path)
    sorted_files = sorted(files)

    # 載入買進價格
    df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
    price_np = df.to_numpy()
    prices = price_np.tolist()  # [時間,價格]

    signal_2 = []

    for price in prices:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(price[0], '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        

        for j, file_name in enumerate(sorted_files):
            if file_name == price[0]:
                with open(folder_path + file_name, encoding='utf-8') as f:
                    
                    general_prompt = f'請依據{p_mutate}對以下關於{stock_names[idx]}的報導，給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷'

                    news = f.read()
                    # print(news)
                    prompt = general_prompt + "\n" + news
                    # gemini的回應，回傳帶有買入訊號的list
                    signal = gemini_response(llm, prompt, regenerate_count)
                    # print(price)
                    # print(signal)
                    sig = signal[1]

                    id.append(price[0])
                    text.append(news)
                    response.append(sig)
                    # [id(日期), text(新聞), llm回應]
                    # signal_2.append([price[0], news, sig])
                break
            
        # print(id)      
        # print(text)      
        # print(response)        

# todo 用pd寫入CSV文件
df = pd.DataFrame({'id': id, 'text': text, 'response': response})
# with open('dataset/id.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(id)

# with open('dataset/text.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(text)

# with open('dataset/response.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(response)

執行1101, 0
執行2211, 1
執行2385, 2
執行2542, 3
執行2880, 4
執行2912, 5
執行3023, 6
執行3264, 7
執行5269, 8
執行8027, 9


In [26]:
# 標籤,分數
import pandas as pd
from collections import Counter

df = pd.read_csv("dataset/signal.csv")
df['text'].tolist()
df['response'].tolist()

text = []
response = []

# 平衡資料 移除部分標籤
for i in range(-100,100):
    for j in range(len(df['response'].tolist())):
        if df['response'].tolist()[j] == i:
            # print(df['response'].tolist()[j])
            response.append(df['response'].tolist()[j])
            text.append(df['text'].tolist()[j])
            break

response2 = response
# 使用Counter計算每個元素的出現次數
element_counts = Counter(response2)
# 將字典轉換為元組列表，並按值進行排序
sorted_items = sorted(element_counts.items(), key=lambda x: x[0])

# 打印結果
print(sorted_items)

[(-100, 1), (-95, 1), (-94, 1), (-91, 1), (-90, 1), (-88, 1), (-87, 1), (-85, 1), (-83, 1), (-82, 1), (-81, 1), (-80, 1), (-78, 1), (-77, 1), (-75, 1), (-74, 1), (-72, 1), (-71, 1), (-70, 1), (-66, 1), (-65, 1), (-64, 1), (-63, 1), (-62, 1), (-61, 1), (-60, 1), (-59, 1), (-57, 1), (-56, 1), (-55, 1), (-54, 1), (-52, 1), (-51, 1), (-50, 1), (-49, 1), (-45, 1), (-44, 1), (-43, 1), (-42, 1), (-41, 1), (-40, 1), (-39, 1), (-38, 1), (-37, 1), (-36, 1), (-35, 1), (-34, 1), (-33, 1), (-32, 1), (-31, 1), (-30, 1), (-29, 1), (-28, 1), (-27, 1), (-26, 1), (-25, 1), (-24, 1), (-23, 1), (-22, 1), (-21, 1), (-20, 1), (-19, 1), (-18, 1), (-17, 1), (-16, 1), (-15, 1), (-14, 1), (-13, 1), (-12, 1), (-11, 1), (-10, 1), (-9, 1), (-8, 1), (-7, 1), (-6, 1), (-5, 1), (-4, 1), (-3, 1), (-2, 1), (-1, 1), (0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (22, 1), (26, 1), (27, 1), (

In [18]:
# 改成5分標籤
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict

df = pd.read_csv("dataset/signal.csv")
text = df['text'].tolist()
response = df['response'].tolist()

for i in range(len(response)):
    res_int = int(response[i])
    if res_int > 50:
        response[i] = "非常好"
    elif res_int > 0:
        response[i] = "好"
    elif res_int < -50:
        response[i] = "非常不好"
    elif res_int < -50:
        response[i] = "不好"
    else:
        response[i] = "一般"
    
print(response)
#平衡資料 移除部分標籤
for i in range(30):
    idx = response.index("一般")
    text.pop(idx)
    response.pop(idx)


['一般', '一般', '一般', '一般', '一般', '一般', '一般', '非常不好', '一般', '一般', '非常不好', '非常不好', '非常不好', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '非常不好', '一般', '非常不好', '一般', '非常好', '一般', '非常不好', '好', '一般', '非常不好', '非常不好', '一般', '非常好', '一般', '一般', '非常不好', '非常不好', '一般', '一般', '好', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '好', '好', '一般', '一般', '非常不好', '一般', '好', '非常不好', '一般', '一般', '一般', '一般', '非常不好', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '非常不好', '非常不好', '非常不好', '非常不好', '一般', '一般', '一般', '一般', '一般', '非常不好', '非常不好', '一般', '非常不好', '非常不好', '一般', '一般', '一般', '一般', '一般', '好', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '非常不好', '非常不好', '非常不好', '一般', '非常不好', '一般', '一般', '一般', '一般', '好', '一般', '一般', '非常不好', '一般', '一般', '一般', '一般', '一般', '一般', '非常不好', '一般', '非常不好', '一般', '一般', '非常不好', '非常不好', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '一般', '好', '非常不好', '一般', '一般', '非常不好', '一般', '一般', '非常不好', '非常好', '非常好', '一般', '非常不好', '一般', '非常不好', '非常不好', '一般',

In [30]:
# 整理上傳hugginface

# 第一次需要安裝
# %pip install transformers
# %pip install huggingface_hub
# %pip install datasets

import datasets
from datasets import Dataset, DatasetDict

print("檢查這兩個值是否相等")
print(len(response))
print(len(text))

train_val_splt = 0.8
trn_tst_spt = 0.9

n = int(0.8*len(response))
n2 = int(0.9*len(response))


tdf = pd.DataFrame({"text": text[:n], "response": response[:n]})
vdf = pd.DataFrame({"text": text[n:n2], "response": response[n:n2]})
ted = pd.DataFrame({"text": text[n2:], "response": response[n2:]})

tds = Dataset.from_pandas(tdf)
vds = Dataset.from_pandas(vdf)
ted = Dataset.from_pandas(ted)

ds = DatasetDict()

ds['train'] = tds
ds['validation'] = vds
ds['test'] = ted

print(ds)

# 上傳 hugginface
ds.push_to_hub(repo_id="Atomuze/stock_news_score", private=True)

142
142
DatasetDict({
    train: Dataset({
        features: ['text', 'response'],
        num_rows: 113
    })
    validation: Dataset({
        features: ['text', 'response'],
        num_rows: 14
    })
    test: Dataset({
        features: ['text', 'response'],
        num_rows: 15
    })
})


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]
c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\datasets--Atomuze--stock_news_score. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


CommitInfo(commit_url='https://huggingface.co/datasets/Atomuze/stock_news_score/commit/ff372a11c4e45f8e25f952a9b4a3636a2c184222', commit_message='Upload dataset', commit_description='', oid='ff372a11c4e45f8e25f952a9b4a3636a2c184222', pr_url=None, pr_revision=None, pr_num=None)

In [6]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
        # print(f"score: {score}, price: {price}, hold: {hold}, balance: {balance}, total: {balance + hold*price}")
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [4]:
asf = [[['20220105', '-2', '47.1'], ['20220114', '-1', '47.85'], ['20220217', '2', '47.15'], ['20220304', '1', '47.8'], ['20220321', '2', '49.1'], ['20220414', '-1', '48.3'], ['20220517', '-1', '41.5'], ['20220520', '1', '41.45'], ['20220602', '-1', '42.0'], ['20220616', '-2', '40.5'], ['20220621', '-1', '39.0'], ['20220622', '-1', '39.65'], ['20220629', '-1', '39.7'], ['20220705', '2', '40.5'], ['20220707', '-1', '40.35'], ['20220713', '-1', '42.0'], ['20220719', '-1', '40.7'], ['20220818', '-1', '39.9'], ['20220826', '-2', '39.7'], ['20220912', '1', '39.95'], ['20220919', '-2', '38.9'], ['20220928', '-1', '35.8'], ['20220929', '1', '34.5'], ['20221011', '-2', '34.1'], ['20221014', '1', '33.0'], ['20221019', '-1', '31.45'], ['20221020', '-1', '31.2'], ['20221026', '-1', '30.0'], ['20221108', '1', '31.15'], ['20221111', '-2', '32.3'], ['20221114', '2', '32.45'], ['20221118', '-2', '33.7'], ['20221122', '1', '32.75'], ['20221124', '-1', '33.0'], ['20221129', '-2', '33.05'], ['20221201', '-1', '34.4'], ['20221202', '-1', '33.9'], ['20221216', '1', '33.2'], ['20221219', '-1', '34.15'], ['20221229', '-2', '34.25'], ['20221230', '-1', '33.9'], ['20230103', '-2', '33.6'], ['20230104', '-2', '33.65'], ['20230106', '-1', '34.45'], ['20230113', '-2', '35.55'], ['20230207', '-1', '36.2'], ['20230222', '1', '38.15'], ['20230224', '-1', '39.65'], ['20230301', '1', '38.85'], ['20230309', '-2', '37.75'], ['20230314', '1', '36.4'], ['20230316', '-1', '36.9'], ['20230317', '-1', '36.6'], ['20230321', '-1', '37.0'], ['20230324', '-2', '37.4'], ['20230327', '-1', '37.0'], ['20230331', '-1', '36.2'], ['20230421', '1', '37.5'], ['20230426', '1', '38.2'], ['20230427', '-2', '39.2'], ['20230502', '2', '39.0'], ['20230526', '-1', '37.8'], ['20230529', '-1', '38.05'], ['20230530', '-2', '38.1'], ['20230601', '-1', '38.05'], ['20230608', '-2', '38.8'], ['20230628', '1', '38.45'], ['20230704', '1', '38.2'], ['20230720', '-1', '36.65'], ['20230721', '-1', '36.6'], ['20230807', '1', '37.5'], ['20230830', '-1', '35.3'], ['20230904', '1', '35.15'], ['20230912', '2', '34.3'], ['20230913', '-2', '34.85'], ['20231011', '-1', '33.15'], ['20231012', '1', '33.2'], ['20231016', '-2', '33.8'], ['20231018', '1', '32.65'], ['20231025', '-1', '31.65'], ['20231027', '-2', '31.55'], ['20231030', '1', '31.8'], ['20231108', '1', '32.8'], ['20231115', '2', '33.0'], ['20231127', '1', '33.9'], ['20231128', '1', '35.0'], ['20231129', '2', '34.6'], ['20231214', '2', '34.5'], ['20231215', '-1', '34.5']], [['20220408', '-1', '62.1'], ['20220419', '-2', '59.2'], ['20220513', '1', '58.5'], ['20220520', '-1', '60.3'], ['20220531', '-1', '62.8'], ['20220610', '-1', '59.9'], ['20220721', '-1', '62.4'], ['20220805', '-2', '62.6'], ['20220929', '-1', '55.3'], ['20221209', '-1', '51.8'], ['20230310', '1', '52.7'], ['20230412', '-1', '56.8'], ['20230511', '-1', '61.0'], ['20230607', '-1', '63.4'], ['20231020', '-1', '69.5'], ['20231130', '-2', '80.9'], ['20231201', '1', '82.0']], [['20220208', '2', '88.0'], ['20220406', '1', '88.6'], ['20220608', '-2', '86.7'], ['20220706', '2', '75.7'], ['20220801', '-1', '79.1'], ['20220808', '2', '78.7'], ['20220823', '-2', '82.6'], ['20221006', '2', '82.9'], ['20221103', '1', '79.3'], ['20221125', '-1', '81.7'], ['20230105', '2', '86.0'], ['20230201', '1', '87.0'], ['20230308', '-1', '94.5'], ['20230309', '2', '95.6'], ['20230406', '1', '94.3'], ['20230424', '-1', '94.6'], ['20230426', '1', '94.3'], ['20230512', '1', '98.0'], ['20230606', '2', '106.0'], ['20230706', '-1', '101.0'], ['20230714', '2', '102.0'], ['20230731', '-1', '107.0'], ['20230901', '-1', '103.5'], ['20230906', '1', '108.0'], ['20231005', '1', '117.0'], ['20231013', '-1', '121.0'], ['20231024', '-1', '124.5'], ['20231103', '-1', '124.0'], ['20231109', '1', '142.0'], ['20231213', '2', '170.0']], [['20220311', '-1', '49.05'], ['20220317', '-1', '47.05'], ['20220624', '-1', '43.55'], ['20220704', '1', '44.15'], ['20220719', '-1', '45.2'], ['20221206', '1', '44.3'], ['20221214', '2', '42.6'], ['20230110', '2', '40.5'], ['20230214', '-1', '41.8'], ['20230301', '1', '42.3'], ['20230307', '-1', '41.75'], ['20230314', '-1', '41.7'], ['20230411', '2', '41.6'], ['20230419', '-2', '42.15'], ['20230511', '2', '41.25'], ['20230518', '1', '42.0'], ['20230524', '-2', '42.75'], ['20230525', '-1', '42.95'], ['20230615', '-1', '42.95'], ['20230801', '2', '42.3'], ['20230921', '-1', '44.05'], ['20231026', '-1', '41.0'], ['20231124', '1', '41.3'], ['20231130', '-1', '40.85'], ['20231228', '1', '40.1']], [['20220126', '-2', '21.65'], ['20220209', '-1', '23.0'], ['20220323', '1', '23.4'], ['20220401', '1', '24.05'], ['20220413', '-1', '25.0'], ['20220503', '-1', '23.6'], ['20220616', '-1', '23.4'], ['20220629', '1', '22.85'], ['20220706', '-1', '22.7'], ['20220712', '-1', '21.7'], ['20220718', '2', '21.7'], ['20220810', '1', '24.0'], ['20220901', '-1', '22.8'], ['20220926', '-1', '22.6'], ['20221012', '-1', '21.6'], ['20221013', '-1', '21.6'], ['20221019', '-1', '20.55'], ['20221104', '-1', '20.65'], ['20221130', '-2', '22.4'], ['20221206', '1', '22.15'], ['20221216', '1', '22.5'], ['20221222', '2', '22.7'], ['20221226', '-1', '22.5'], ['20230109', '2', '22.95'], ['20230111', '-1', '23.45'], ['20230201', '-1', '22.95'], ['20230209', '2', '22.55'], ['20230215', '-1', '22.85'], ['20230328', '1', '22.1'], ['20230526', '1', '22.2'], ['20230531', '2', '22.35'], ['20230619', '-1', '22.2'], ['20230621', '-2', '22.25'], ['20230817', '-2', '21.0'], ['20230905', '1', '20.65'], ['20230921', '1', '20.85'], ['20231025', '-1', '20.5'], ['20231030', '1', '20.45'], ['20231127', '1', '21.9'], ['20231206', '-1', '22.05']], [['20220105', '-2', '272.0'], ['20220111', '-1', '272.5'], ['20220112', '-1', '274.0'], ['20220118', '-2', '269.0'], ['20220119', '2', '270.5'], ['20220125', '2', '265.5'], ['20220126', '-1', '266.0'], ['20220211', '1', '264.5'], ['20220222', '-1', '260.5'], ['20220224', '1', '261.0'], ['20220309', '1', '264.0'], ['20220324', '-2', '261.5'], ['20220325', '-1', '262.0'], ['20220331', '-1', '263.5'], ['20220415', '2', '267.0'], ['20220511', '-1', '264.0'], ['20220608', '-2', '265.5'], ['20220623', '1', '268.0'], ['20220627', '1', '274.0'], ['20220719', '-1', '289.0'], ['20220720', '-2', '285.5'], ['20220802', '2', '282.0'], ['20220804', '2', '276.5'], ['20220816', '2', '279.5'], ['20220824', '2', '276.5'], ['20220826', '2', '273.0'], ['20220905', '1', '272.5'], ['20220912', '1', '284.0'], ['20220914', '-1', '283.5'], ['20220920', '1', '273.5'], ['20220921', '1', '277.5'], ['20221004', '-2', '282.0'], ['20221107', '2', '263.0'], ['20221110', '-2', '265.0'], ['20221201', '-1', '274.0'], ['20221228', '-1', '269.5'], ['20230106', '-1', '273.0'], ['20230113', '1', '271.5'], ['20230210', '2', '267.0'], ['20230213', '-2', '269.5'], ['20230214', '-1', '271.0'], ['20230216', '1', '269.0'], ['20230217', '1', '269.0'], ['20230221', '1', '268.5'], ['20230222', '1', '268.5'], ['20230223', '-1', '269.0'], ['20230224', '1', '270.0'], ['20230302', '1', '267.0'], ['20230306', '1', '269.0'], ['20230308', '-1', '269.5'], ['20230315', '2', '268.5'], ['20230323', '-1', '269.5'], ['20230331', '-1', '270.0'], ['20230421', '1', '270.5'], ['20230427', '1', '271.5'], ['20230428', '1', '272.5'], ['20230505', '-1', '283.5'], ['20230511', '2', '284.5'], ['20230601', '2', '280.0'], ['20230612', '1', '283.5'], ['20230614', '-1', '285.0'], ['20230620', '2', '287.0'], ['20230627', '-1', '283.0'], ['20230710', '-1', '282.0'], ['20230719', '-2', '284.5'], ['20230818', '2', '272.5'], ['20230911', '1', '267.5'], ['20230927', '2', '263.5'], ['20231002', '1', '263.0'], ['20231013', '1', '262.0'], ['20231016', '1', '263.0'], ['20231031', '2', '256.5'], ['20231101', '1', '258.5'], ['20231106', '2', '264.0'], ['20231127', '2', '271.0'], ['20231205', '2', '269.5'], ['20231208', '2', '270.5'], ['20231215', '-2', '267.0'], ['20231218', '-1', '267.0']], [['20220207', '-1', '261.5'], ['20220208', '2', '269.5'], ['20220310', '2', '265.0'], ['20220531', '-2', '264.0'], ['20220712', '-1', '245.0'], ['20220905', '1', '283.5'], ['20221012', '-1', '279.0'], ['20221024', '1', '294.0'], ['20221208', '2', '268.0'], ['20221228', '1', '276.0'], ['20230103', '-2', '275.0'], ['20230201', '-1', '291.0'], ['20230207', '-1', '312.0'], ['20230222', '-1', '296.0'], ['20230309', '-2', '303.0'], ['20230323', '-1', '307.0'], ['20230420', '-2', '333.5'], ['20230508', '-1', '350.5'], ['20230605', '1', '356.0'], ['20230607', '-1', '359.5'], ['20230714', '-1', '377.0'], ['20230721', '-1', '334.0'], ['20230904', '-1', '312.5'], ['20230921', '1', '301.0'], ['20230925', '-2', '306.5'], ['20231002', '2', '325.0'], ['20231024', '-1', '305.5'], ['20231206', '1', '287.5']], [['20220121', '1', '52.9'], ['20220408', '1', '48.4'], ['20230224', '1', '56.8'], ['20230313', '-2', '56.3'], ['20230525', '1', '54.9'], ['20230616', '1', '64.8'], ['20230726', '-1', '62.7'], ['20231026', '-1', '63.9'], ['20231030', '-2', '64.1'], ['20231206', '-1', '77.2'], ['20231211', '1', '78.4']], [['20220120', '1', '1585'], ['20220125', '-2', '1600'], ['20220207', '-1', '1585'], ['20220224', '-1', '1715'], ['20220303', '-1', '1840'], ['20220310', '2', '1800'], ['20220328', '2', '1915'], ['20220422', '2', '1590'], ['20220518', '1', '1380'], ['20220630', '-1', '1140'], ['20220815', '2', '938'], ['20220908', '-1', '826'], ['20221122', '-2', '738'], ['20230309', '-2', '1005'], ['20230410', '-1', '1190'], ['20230412', '-2', '1300'], ['20230511', '1', '1120'], ['20230512', '2', '1120'], ['20230515', '-1', '1160'], ['20230517', '2', '1200'], ['20230518', '-2', '1205'], ['20230621', '1', '1170'], ['20230808', '-1', '978'], ['20230824', '-1', '907'], ['20230825', '-1', '914'], ['20230921', '-1', '1160'], ['20231120', '2', '1525']], [['20220418', '-1', '70.8'], ['20220425', '1', '74.1'], ['20220503', '-1', '74.8'], ['20220520', '1', '76.9'], ['20220919', '-1', '62.9'], ['20221005', '-1', '56.5'], ['20221104', '-2', '53.0'], ['20230524', '-2', '46.2'], ['20230606', '-1', '55.6'], ['20230615', '-2', '62.2'], ['20230616', '1', '59.9'], ['20230704', '1', '57.4'], ['20230808', '1', '78.8'], ['20230811', '1', '68.0'], ['20230919', '1', '67.9'], ['20230925', '1', '71.1'], ['20231003', '-1', '71.8'], ['20231205', '2', '66.3']]]

In [8]:
import test
list = asf

# print(list[3])
rtns = []
bnhs = []

for i in range(len(list)):
    rtn, bnh = calculate_return(list[i], 0)
    print(rtn, bnh)
    rtns.append(rtn)
    bnhs.append(bnh)
print(f'平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{sum(bnhs) / len(bnhs)}')
# raw_data.append(signal_2_np.tolist())

1.07087 0.73248
1.05487 1.32045
1.43148 1.93182
1.0522 0.81753
1.16377 1.01848
1.02707 0.98162
0.90841 1.09943
1.10544 1.48204
0.53451 0.96215
1.00397 0.93644
平均報酬率:1.0352590000000002，平均總額投入報酬率:1.128244
